# Hybrid vs Individual Model Comparison (RF vs XGBoost vs LightGBM vs Hybrid)

This notebook compares RandomForest, XGBoost, LightGBM and a hybrid ensemble.

If `xgboost` or `lightgbm` are not installed, it uses gradient-boosting proxies with matching role names so the workflow still runs.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split



In [ ]:
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGBClassifier = GradientBoostingClassifier
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBMClassifier = HistGradientBoostingClassifier
    LGBM_AVAILABLE = False

print('xgboost available:', XGB_AVAILABLE)
print('lightgbm available:', LGBM_AVAILABLE)



In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
for df in (train_df, test_df):
    if '' in df.columns:
        df.drop(columns=[''], inplace=True)

X_train_full = train_df.drop(columns=['Converted'])
y_train_full = train_df['Converted'].astype(int)
X_test = test_df.drop(columns=['Converted'])
y_test = test_df['Converted'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)



In [ ]:
def evaluate_binary(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }

models = {
    'RandomForest': RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42),
}

if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(
        n_estimators=80,
        max_depth=3,
        learning_rate=0.08,
        eval_metric='logloss',
        random_state=13,
    )
else:
    models['XGBoost'] = XGBClassifier(
        n_estimators=80,
        learning_rate=0.08,
        max_depth=3,
        random_state=13,
    )

if LGBM_AVAILABLE:
    models['LightGBM'] = LGBMClassifier(
        n_estimators=160,
        max_depth=6,
        learning_rate=0.07,
        random_state=7,
    )
else:
    models['LightGBM'] = LGBMClassifier(
        max_iter=160,
        max_depth=6,
        learning_rate=0.07,
        random_state=7,
    )

model_info = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    val_proba = model.predict_proba(X_val)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]
    model_info[name] = {
        'val_proba': val_proba,
        'test_proba': test_proba,
        'val_metrics': evaluate_binary(y_val.values, val_proba),
        'test_metrics': evaluate_binary(y_test.values, test_proba),
    }

print('Individual model training complete.')



In [ ]:
# Tune hybrid weights on validation accuracy/f1/auc
best = None
for wr in np.arange(0, 1.01, 0.1):
    for wx in np.arange(0, 1.01, 0.1):
        wl = 1.0 - wr - wx
        if wl < 0:
            continue

        val_hybrid = (
            wr * model_info['RandomForest']['val_proba'] +
            wx * model_info['XGBoost']['val_proba'] +
            wl * model_info['LightGBM']['val_proba']
        )
        m = evaluate_binary(y_val.values, val_hybrid)
        score = (m['accuracy'], m['f1'], m['roc_auc'])
        if best is None or score > best['score']:
            best = {'weights': (wr, wx, wl), 'score': score, 'val_metrics': m}

wr, wx, wl = best['weights']
test_hybrid = (
    wr * model_info['RandomForest']['test_proba'] +
    wx * model_info['XGBoost']['test_proba'] +
    wl * model_info['LightGBM']['test_proba']
)
hybrid_test = evaluate_binary(y_test.values, test_hybrid)

print('Best weights (RF, XGB, LGBM):', best['weights'])



In [ ]:
rows = []
for name, info in model_info.items():
    test_m = info['test_metrics']
    predictive_efficiency = (test_m['accuracy'] + test_m['f1'] + test_m['roc_auc']) / 3
    rows.append({
        'model': name,
        **{k: round(v, 4) for k, v in test_m.items()},
        'predictive_efficiency': round(predictive_efficiency, 4),
    })

hybrid_eff = (hybrid_test['accuracy'] + hybrid_test['f1'] + hybrid_test['roc_auc']) / 3
rows.append({
    'model': 'HybridEnsemble',
    **{k: round(v, 4) for k, v in hybrid_test.items()},
    'predictive_efficiency': round(hybrid_eff, 4),
})

comparison_df = pd.DataFrame(rows).sort_values(['accuracy', 'predictive_efficiency'], ascending=False).reset_index(drop=True)
print(comparison_df.to_string(index=False))

best_row = comparison_df.iloc[0]
print('\nBest model on this benchmark:', best_row['model'])
print('Best accuracy:', best_row['accuracy'])

